# In2STEM Project 1: Star Formation Rates and the Galaxy Main Sequence

In this project you will work with DESI galaxy data. You will use emission-line measurements to calculate star formation rates (SFRs), then test whether the galaxies follow the star-forming main sequence.

This is the student version. Some code cells contain `...` for you to replace.


## The Science Idea

Galaxies form stars from cold gas. A galaxy with a high star formation rate is making many new stars each year, while a galaxy with a low star formation rate is forming stars more slowly.

We can estimate star formation rate using the H$\alpha$ emission line. Young, massive stars ionise gas around them. When that gas recombines, it emits light at specific wavelengths, including H$\alpha$ and H$\beta$.

Dust absorbs some of this light, so we first correct for dust attenuation. Then we convert corrected H$\alpha$ luminosity into a star formation rate.


In [ ]:
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
import numpy as np
import seaborn as sns

%matplotlib inline

cosmo = FlatLambdaCDM(H0=70 * u.km / u.s / u.Mpc, Tcmb0=2.725 * u.K, Om0=0.3)


In [ ]:
# Load the low-redshift DESI teaching sample.
# The relative path works both locally and on Binder.
file_path = 'data/low_z_sample.csv'
data = pd.read_csv(file_path)


In [ ]:
# Display a preview of the table.
data


In [ ]:
# These are the columns we will use most often.
important_columns = ['Z', 'HALPHA_FLUX', 'HALPHA_BD', 'LOGMSTAR']
data[important_columns][:5]


## Day 1: Calculating Star Formation Rates

The two main columns for the first calculation are:

- `HALPHA_FLUX`: the observed H$\alpha$ flux.
- `HALPHA_BD`: the Balmer decrement, which is the ratio $F_{H\alpha}/F_{H\beta}$.

The Balmer decrement tells us how much dust has dimmed the H$\alpha$ light.


### 1. Correct for Dust Attenuation

We calculate the H$\alpha$ attenuation using:

$$ A(H\alpha) = \frac{2.5 \log_{10}\left(\frac{1}{2.86} \times \frac{F_{H\alpha}}{F_{H\beta}}\right)}{\frac{k(H\beta)}{k(H\alpha)} - 1} $$

For this project we use $k(H\beta)/k(H\alpha) = 1.53$ from Calzetti et al. (2000). The flux ratio is already saved as `data['HALPHA_BD']`.


In [ ]:
# TODO: calculate H-alpha attenuation using the equation above.
# Hint: np.log10(...) calculates log base 10.
h_alpha_attenuation = ...

data['HALPHA_ATTENUATION'] = h_alpha_attenuation

print('Maximum H-alpha attenuation:', np.nanmax(h_alpha_attenuation))


In [ ]:
sns.histplot(data['HALPHA_ATTENUATION'], bins=100, color='steelblue', edgecolor='white')
plt.xlabel(r'H$\alpha$ attenuation')
plt.ylabel('Number of galaxies')
plt.xlim(0, 20)
plt.show()


### 2. Convert Flux into Luminosity

Flux is how bright an object appears from Earth. Luminosity is how much light the object is really producing.

To convert flux into luminosity, we need the luminosity distance, which we estimate from redshift:

$$ L(H\alpha)_{obs} = 4\pi d_L^2 F_{H\alpha} $$

The H$\alpha$ flux values in the table are stored in units of $10^{-17}$ erg cm$^{-2}$ s$^{-1}$, so we multiply by $10^{-17}$ before calculating luminosity.


In [ ]:
# Calculate luminosity distance in cm.
dlum_cm = cosmo.luminosity_distance(np.asarray(data['Z'])).to(u.cm).value

# Convert H-alpha flux into erg / cm^2 / s.
h_alpha_flux = np.asarray(data['HALPHA_FLUX']) * 1e-17

# TODO: calculate observed H-alpha luminosity.
luminosity = ...

data['HALPHA_LUMINOSITY_OBSERVED'] = luminosity

print('Maximum observed luminosity:', np.nanmax(luminosity))


### 3. Apply the Dust Correction

Now we correct the observed luminosity for dust attenuation:

$$ L(H\alpha)_{corr} = L(H\alpha)_{obs} \times 10^{0.4A(H\alpha)} $$


In [ ]:
# TODO: calculate the dust-corrected H-alpha luminosity.
luminosity_corrected = ...

data['HALPHA_LUMINOSITY'] = luminosity_corrected

print('Maximum corrected luminosity:', np.nanmax(luminosity_corrected))


### 4. Calculate Star Formation Rate

We convert corrected H$\alpha$ luminosity into star formation rate using:

$$ SFR[M_{\odot}\ yr^{-1}] = (5.5 \times 10^{-42}) \times L(H\alpha)_{corr} $$

The units mean "solar masses of new stars formed per year".


In [ ]:
# TODO: calculate the star formation rate.
sfr = ...

data['SFR_CALCULATED'] = sfr

sfr[:10]


In [ ]:
sns.histplot(data['SFR_CALCULATED'], bins=100, color='seagreen', edgecolor='white')
plt.xlabel(r'Star formation rate [$M_\odot$ yr$^{-1}$]')
plt.ylabel('Number of galaxies')
plt.xlim(0, 20)
plt.title('Distribution of Star Formation Rates')
plt.show()


### Questions

1. What kind of SFR values do most galaxies have?
2. What does an SFR of 1 $M_{\odot}$ yr$^{-1}$ mean in words?
3. Are the mean and median similar? What does that tell you about the shape of the distribution?


In [ ]:
# TODO: calculate the mean and median SFR.
mean_sfr = ...
median_sfr = ...

print('Mean SFR:', mean_sfr)
print('Median SFR:', median_sfr)


In [ ]:
# Make a safe log(SFR) column for plotting.
sfr_values = np.asarray(data['SFR_CALCULATED'])
log_sfr = np.full(len(data), np.nan)
positive_sfr = np.isfinite(sfr_values) & (sfr_values > 0)
log_sfr[positive_sfr] = np.log10(sfr_values[positive_sfr])

data['LOG_SFR'] = log_sfr

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(
    data['Z'],
    data['LOG_SFR'],
    c=data['LOGMSTAR'],
    cmap='turbo',
    s=8,
    alpha=0.7
)
ax.set_xlabel('Redshift, z')
ax.set_ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.colorbar(scatter, label=r'log($M_* / M_\odot$)', ax=ax)
plt.show()


### Explore the Data

Try plotting different quantities against each other. For example, you could compare SFR, stellar mass, redshift, luminosity, or attenuation.


In [ ]:
# Change x and y to explore different quantities.
x = data['LOGMSTAR']
y = data['LOG_SFR']

plt.figure(figsize=(8, 6))
plt.plot(x, y, '.', markersize=4, alpha=0.5)
plt.xlabel(r'log($M_* / M_\odot$)')
plt.ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.show()


## Day 2: The Star-Forming Main Sequence

Astronomers have found that many star-forming galaxies follow a relationship between stellar mass and star formation rate. This is called the **star-forming main sequence**.

In simple terms:

- more massive star-forming galaxies usually have higher SFRs;
- galaxies far above the relation may be having a starburst;
- galaxies far below the relation may be forming stars less efficiently or starting to quench.

We will plot $\log(SFR)$ against $\log(M_*)$, fit a straight line, and compare each galaxy to that line.


### 5. Plot log(SFR) Against log(Stellar Mass)

The table already contains `LOGMSTAR`, which is $\log(M_*/M_{\odot})$.

We calculated `LOG_SFR` above. We only fit galaxies where mass and SFR are both valid numbers.


In [ ]:
log_mass = np.asarray(data['LOGMSTAR'])
log_sfr = np.asarray(data['LOG_SFR'])

# TODO: make a mask that is True only when both log_mass and log_sfr are finite numbers.
valid_main_sequence = ...

plt.figure(figsize=(8, 6))
plt.scatter(log_mass[valid_main_sequence], log_sfr[valid_main_sequence], s=8, alpha=0.35)
plt.xlabel(r'log($M_* / M_\odot$)')
plt.ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.title('Star Formation Rate vs Stellar Mass')
plt.show()


### 6. Fit a Simple Main-Sequence Relation

We will fit a straight line:

$$ \log(SFR) = m \log(M_*) + c $$

where $m$ is the gradient and $c$ is the intercept. This is not a perfect scientific model, but it is a good first test for this project.


In [ ]:
# TODO: fit a straight line to log_sfr against log_mass.
fit_gradient, fit_intercept = np.polyfit(
    ...,
    ...,
    deg=1
)

print('Best-fitting relation:')
print(f'log(SFR) = {fit_gradient:.3f} log(M*) + {fit_intercept:.3f}')

mass_grid = np.linspace(np.nanmin(log_mass[valid_main_sequence]), np.nanmax(log_mass[valid_main_sequence]), 100)
fit_line = fit_gradient * mass_grid + fit_intercept

plt.figure(figsize=(8, 6))
plt.scatter(log_mass[valid_main_sequence], log_sfr[valid_main_sequence], s=8, alpha=0.25, label='Galaxies')
plt.plot(mass_grid, fit_line, color='black', linewidth=2.5, label='Best-fitting line')
plt.xlabel(r'log($M_* / M_\odot$)')
plt.ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.legend()
plt.show()


### 7. Predict the Main-Sequence SFR and Classify Galaxies

For each galaxy, we can use the fitted line to predict the SFR we would expect if it sat exactly on the main sequence.

We then calculate:

$$ \Delta \log(SFR) = \log(SFR)_{observed} - \log(SFR)_{main\ sequence} $$

For this project we will use a simple rule:

- `above main sequence`: more than 0.3 dex above the fitted line;
- `main sequence`: within 0.3 dex of the fitted line;
- `below main sequence`: more than 0.3 dex below the fitted line.

A difference of 0.3 dex is about a factor of 2 in SFR.


In [ ]:
# TODO: use the fitted relation to calculate the expected log(SFR).
main_sequence_log_sfr = ...

# TODO: convert expected log(SFR) back into expected SFR.
main_sequence_sfr = ...

# TODO: calculate observed minus expected log(SFR).
main_sequence_offset = ...

classification = np.full(len(data), 'main sequence', dtype=object)

# TODO: classify galaxies above and below the main sequence using a 0.3 dex threshold.
classification[main_sequence_offset > ...] = 'above main sequence'
classification[main_sequence_offset < ...] = 'below main sequence'
classification[~valid_main_sequence] = 'not classified'

data['MS_LOG_SFR_EXPECTED'] = main_sequence_log_sfr
data['MS_SFR_EXPECTED'] = main_sequence_sfr
data['MS_OFFSET'] = main_sequence_offset
data['MS_CLASS'] = classification


In [ ]:
# Run this after you have classified the galaxies.
classes, counts = np.unique(data['MS_CLASS'], return_counts=True)

for label, count in zip(classes, counts):
    print(f'{label}: {count}')


In [ ]:
colour_map = {
    'below main sequence': 'royalblue',
    'main sequence': 'darkgreen',
    'above main sequence': 'crimson'
}

plt.figure(figsize=(8, 6))

for label, colour in colour_map.items():
    mask = np.asarray(data['MS_CLASS']) == label
    plt.scatter(log_mass[mask], log_sfr[mask], s=10, alpha=0.45, color=colour, label=label)

plt.plot(mass_grid, fit_line, color='black', linewidth=2.5, label='Best-fitting line')
plt.xlabel(r'log($M_* / M_\odot$)')
plt.ylabel(r'log(SFR / $M_\odot$ yr$^{-1}$)')
plt.legend()
plt.show()


In [ ]:
sns.histplot(data['MS_OFFSET'][valid_main_sequence], bins=60, color='mediumpurple', edgecolor='white')
plt.axvline(-0.3, color='black', linestyle='--')
plt.axvline(0.3, color='black', linestyle='--')
plt.xlabel(r'$\Delta$ log(SFR) from fitted main sequence')
plt.ylabel('Number of galaxies')
plt.show()


## Final Discussion Questions

1. Does the fitted relation show that more massive galaxies tend to have higher SFRs?
2. Which class has the most galaxies: below, on, or above the main sequence?
3. What might cause a galaxy to sit above the main sequence?
4. What might cause a galaxy to sit below the main sequence?
5. How could the classification change if we used a different threshold, such as 0.2 dex or 0.5 dex?
